# Essential Final Assignment: GPA-Change Regression

## Problem statement

This notebook predicts how much a student's GPA changed during the semester:

`GPA_Change = Post_Semester_GPA - Pre_Semester_GPA`

The main research question is:

> Do AI-usage variables add useful predictive information about GPA change beyond previous GPA and general study context?

This is a **regression** problem because GPA change is a continuous numeric value.

## Evidence boundary

The dataset does not contain a credible non-AI control group, random assignment, documented sampling, or confirmed real-world provenance. The results can show predictive associations, but they cannot prove that AI usage caused GPA to increase or decrease.

## 1. Aim and objectives

**Aim:** Build and critically evaluate regression models for semester GPA change.

**Objectives:**

1. Understand and prepare the dataset without changing the raw CSV.
2. Compare a context-only feature set with one that adds AI-usage variables.
3. Compare at least three regression models using five-fold cross-validation.
4. Tune the strongest nonlinear model using training data only.
5. Evaluate the models on an untouched 20% test set.
6. Explain the results, errors, important features, limitations, and practical meaning.

## 2. Setup and data loading

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5

def find_dataset():
    relative = Path("Datasets/Impact of AI on Students/ai_student_impact_dataset.csv")
    for start in [Path.cwd(), *Path.cwd().parents]:
        candidate = start / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {relative}")

DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from:")
print(DATA_PATH)

Loaded 50,000 rows and 16 columns from:
D:\APU Study\1st Semester\Applied Machine Learning\AML Assignment\Datasets\Impact of AI on Students\ai_student_impact_dataset.csv


## 3. Dataset understanding and essential EDA

This section checks whether the file loaded correctly, creates the target, and examines its distribution. It also checks relationships that may help explain model behaviour.

In [2]:
df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]

audit = pd.Series({
    "Rows": len(df),
    "Original columns": 16,
    "Missing cells": int(df.iloc[:, :16].isna().sum().sum()),
    "Duplicate rows": int(df.iloc[:, :16].duplicated().sum()),
    "Unique student IDs": int(df["Student_ID"].nunique()),
    "Students with GPA increase": int((df["GPA_Change"] > 0).sum()),
    "Students with GPA decrease": int((df["GPA_Change"] < 0).sum()),
})
display(audit.to_frame("Value"))

target_summary = df["GPA_Change"].describe().rename("GPA Change")
display(target_summary.to_frame())

assert df["Student_ID"].nunique() == len(df)
assert df.iloc[:, :16].isna().sum().sum() == 0

,Value
Rows,50000
Original columns,16
Missing cells,0
Duplicate rows,0
Unique student IDs,50000
Students with GPA increase,43759
Students with GPA decrease,6192


,GPA Change
count,50000.000000
mean,0.203197
std,0.187192
min,-0.924000
25%,0.087000
50%,0.204000
75%,0.325000
max,1.008000


In [3]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.histplot(df["GPA_Change"], bins=40, kde=True, ax=axes[0, 0])
axes[0, 0].axvline(0, color="black", linestyle="--")
axes[0, 0].set_title("Distribution of semester GPA change")

sample = df.sample(5000, random_state=RANDOM_STATE)
sns.scatterplot(
    data=sample, x="Pre_Semester_GPA", y="GPA_Change",
    alpha=0.25, ax=axes[0, 1]
)
axes[0, 1].set_title("Previous GPA and GPA change")

sns.boxplot(
    data=df, x="Prompt_Engineering_Skill", y="GPA_Change",
    order=["Beginner", "Intermediate", "Advanced"], ax=axes[1, 0]
)
axes[1, 0].set_title("GPA change by prompt skill")

df["AI_Hours_Quartile"] = pd.qcut(
    df["Weekly_GenAI_Hours"], 4,
    labels=["Lowest", "Low-Medium", "High-Medium", "Highest"]
)
sns.pointplot(
    data=df, x="AI_Hours_Quartile", y="GPA_Change",
    errorbar=("ci", 95), ax=axes[1, 1]
)
axes[1, 1].set_title("Mean GPA change by weekly AI-hours quartile")

fig.tight_layout()
plt.show()

display(
    df.groupby("Prompt_Engineering_Skill", observed=True)["GPA_Change"]
    .agg(["count", "mean", "std"])
    .round(4)
)
display(
    df.groupby("AI_Hours_Quartile", observed=True)["GPA_Change"]
    .agg(["count", "mean", "std"])
    .round(4)
)

,count,mean,std
Prompt_Engineering_Skill,,,
Advanced,13809,0.2481,0.1987
Beginner,18495,0.1852,0.1799
Intermediate,17696,0.1869,0.1795


,count,mean,std
AI_Hours_Quartile,,,
Lowest,12510,0.1881,0.1753
Low-Medium,12516,0.2055,0.1760
High-Medium,12479,0.2287,0.1755
Highest,12495,0.1905,0.2159


## 4. Data preparation and leakage control

Two feature groups are defined:

- **Context features:** previous GPA, major, year of study, traditional study hours, and exam anxiety.
- **AI features:** weekly AI hours, use case, prompt skill, tool diversity, subscription, dependency, and institutional AI policy.

The comparison between context-only and full features tests whether AI variables add predictive value. `Student_ID`, `Post_Semester_GPA`, `Skill_Retention_Score`, and `Burnout_Risk_Level` are excluded to prevent meaningless identifiers and outcome leakage.

In [4]:
CONTEXT_FEATURES = [
    "Pre_Semester_GPA",
    "Major_Category",
    "Year_of_Study",
    "Traditional_Study_Hours",
    "Anxiety_Level_During_Exams",
]

AI_FEATURES = [
    "Weekly_GenAI_Hours",
    "Primary_Use_Case",
    "Prompt_Engineering_Skill",
    "Tool_Diversity",
    "Paid_Subscription",
    "Perceived_AI_Dependency",
    "Institutional_Policy",
]

FULL_FEATURES = CONTEXT_FEATURES + AI_FEATURES
FORBIDDEN_FEATURES = {
    "Student_ID", "Post_Semester_GPA",
    "Skill_Retention_Score", "Burnout_Risk_Level",
}

assert not FORBIDDEN_FEATURES.intersection(FULL_FEATURES)
assert len(FULL_FEATURES) == len(set(FULL_FEATURES))

display(pd.DataFrame({
    "Feature group": ["Context only", "AI usage", "Full model"],
    "Number of features": [
        len(CONTEXT_FEATURES), len(AI_FEATURES), len(FULL_FEATURES)
    ],
    "Purpose": [
        "Prediction without direct AI-usage variables",
        "AI behaviour and institutional AI context",
        "Context plus AI variables",
    ],
}))

,Feature group,Number of features,Purpose
0,Context only,5,Prediction without direct AI-usage variables
1,AI usage,7,AI behaviour and institutional AI context
2,Full model,12,Context plus AI variables


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

def make_preprocessor(frame):
    categorical = [
        column for column in frame.columns
        if pd.api.types.is_string_dtype(frame[column])
        or pd.api.types.is_bool_dtype(frame[column])
    ]
    numeric = [column for column in frame.columns if column not in categorical]

    return ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric,
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "onehot",
                    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                ),
            ]),
            categorical,
        ),
    ])

train_index, test_index = train_test_split(
    np.arange(len(df)),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

X_train = df.iloc[train_index][FULL_FEATURES]
X_test = df.iloc[test_index][FULL_FEATURES]
y_train = df.iloc[train_index]["GPA_Change"]
y_test = df.iloc[test_index]["GPA_Change"]

print(f"Training rows: {len(X_train):,}")
print(f"Untouched test rows: {len(X_test):,}")

Training rows: 40,000
Untouched test rows: 10,000


## 5. Do AI variables add predictive value?

The same nonlinear model is tested with context-only, AI-only, and full feature sets. If the full model performs better than the context-only model, AI variables contain additional predictive information. This still does not prove causation.

In [6]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

feature_sets = {
    "Context only": CONTEXT_FEATURES,
    "AI variables only": AI_FEATURES,
    "Context + AI variables": FULL_FEATURES,
}

feature_set_rows = []
for feature_set_name, columns in feature_sets.items():
    X_feature_train = df.iloc[train_index][columns]
    X_feature_test = df.iloc[test_index][columns]

    feature_model = Pipeline([
        ("preprocess", make_preprocessor(X_feature_train)),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=150,
                learning_rate=0.08,
                max_leaf_nodes=31,
                random_state=RANDOM_STATE,
            ),
        ),
    ])

    scores = cross_validate(
        feature_model,
        X_feature_train,
        y_train,
        cv=cv,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2",
        },
        n_jobs=-1,
    )
    feature_model.fit(X_feature_train, y_train)
    prediction = feature_model.predict(X_feature_test)

    feature_set_rows.append({
        "Feature set": feature_set_name,
        "CV MAE": -scores["test_MAE"].mean(),
        "CV RMSE": -scores["test_RMSE"].mean(),
        "CV R2": scores["test_R2"].mean(),
        "Test MAE": mean_absolute_error(y_test, prediction),
        "Test RMSE": mean_squared_error(y_test, prediction) ** 0.5,
        "Test R2": r2_score(y_test, prediction),
    })

feature_set_results = pd.DataFrame(feature_set_rows).set_index("Feature set")
display(feature_set_results.style.format("{:.4f}"))

,CV MAE,CV RMSE,CV R2,Test MAE,Test RMSE,Test R2
Feature set,,,,,,
Context only,0.1254,0.1638,0.2378,0.1239,0.1618,0.2382
AI variables only,0.1356,0.1708,0.1708,0.1347,0.1689,0.1703
Context + AI variables,0.1129,0.1443,0.4084,0.1114,0.1415,0.4170


## 6. Model comparison

A mean baseline and four real regression models are compared using the same five folds. Lower MAE/RMSE is better; higher R² is better.

In [7]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Ridge regression": Ridge(alpha=1.0),
    "Random forest": RandomForestRegressor(
        n_estimators=180,
        min_samples_leaf=3,
        max_features=0.8,
        n_jobs=1,
        random_state=RANDOM_STATE,
    ),
    "Histogram gradient boosting": HistGradientBoostingRegressor(
        max_iter=180,
        learning_rate=0.07,
        max_leaf_nodes=31,
        random_state=RANDOM_STATE,
    ),
}

cv_rows = []
fitted_models = {}
for model_name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocess", make_preprocessor(X_train)),
        ("model", estimator),
    ])
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2",
        },
        n_jobs=-1,
    )
    pipeline.fit(X_train, y_train)
    fitted_models[model_name] = pipeline
    cv_rows.append({
        "Model": model_name,
        "CV MAE": -scores["test_MAE"].mean(),
        "CV RMSE": -scores["test_RMSE"].mean(),
        "CV R2": scores["test_R2"].mean(),
        "R2 standard deviation": scores["test_R2"].std(ddof=1),
    })

cv_results = (
    pd.DataFrame(cv_rows)
    .sort_values("CV RMSE")
    .reset_index(drop=True)
)
display(cv_results.style.format({
    "CV MAE": "{:.4f}",
    "CV RMSE": "{:.4f}",
    "CV R2": "{:.4f}",
    "R2 standard deviation": "{:.4f}",
}))

,Model,CV MAE,CV RMSE,CV R2,R2 standard deviation
0,Histogram gradient boosting,0.1130,0.1443,0.4080,0.0085
1,Random forest,0.1149,0.1473,0.3833,0.0122
2,Ridge regression,0.1261,0.1608,0.2657,0.0104
3,Linear regression,0.1261,0.1608,0.2657,0.0104
4,Mean baseline,0.1459,0.1876,-0.0003,0.0004


## 7. Hyperparameter tuning

Histogram gradient boosting is tuned because it handles nonlinear relationships efficiently. Randomized search tests several configurations using only the training folds.

In [8]:
from sklearn.model_selection import RandomizedSearchCV

tuning_pipeline = Pipeline([
    ("preprocess", make_preprocessor(X_train)),
    (
        "model",
        HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    ),
])

tuning_space = {
    "model__learning_rate": [0.03, 0.05, 0.08, 0.10],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__min_samples_leaf": [10, 20, 40],
    "model__l2_regularization": [0.0, 0.1, 1.0],
    "model__max_iter": [120, 180, 240],
}

tuning_search = RandomizedSearchCV(
    tuning_pipeline,
    param_distributions=tuning_space,
    n_iter=10,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
)
tuning_search.fit(X_train, y_train)

tuned_cv_rmse = -tuning_search.best_score_
print("Best parameters:")
display(tuning_search.best_params_)
print(f"Best five-fold CV RMSE: {tuned_cv_rmse:.4f}")

Best parameters:


{'model__min_samples_leaf': 10,
 'model__max_leaf_nodes': 15,
 'model__max_iter': 180,
 'model__learning_rate': 0.05,
 'model__l2_regularization': 0.1}

Best five-fold CV RMSE: 0.1441


## 8. Final test-set evaluation

The untouched test set is now used to compare final generalisation. The tuned model is included alongside the original models.

In [9]:
fitted_models["Tuned histogram gradient boosting"] = tuning_search.best_estimator_

test_rows = []
predictions = {}
for model_name, model in fitted_models.items():
    prediction = model.predict(X_test)
    predictions[model_name] = prediction
    test_rows.append({
        "Model": model_name,
        "Test MAE": mean_absolute_error(y_test, prediction),
        "Test RMSE": mean_squared_error(y_test, prediction) ** 0.5,
        "Test R2": r2_score(y_test, prediction),
    })

test_results = (
    pd.DataFrame(test_rows)
    .sort_values("Test RMSE")
    .reset_index(drop=True)
)
display(test_results.style.format({
    "Test MAE": "{:.4f}",
    "Test RMSE": "{:.4f}",
    "Test R2": "{:.4f}",
}))

best_model_name = test_results.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
best_prediction = predictions[best_model_name]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(
    data=test_results,
    y="Model", x="Test RMSE",
    color="#4C78A8", ax=axes[0]
)
axes[0].set_title("Test RMSE comparison (lower is better)")

sns.scatterplot(
    x=y_test, y=best_prediction,
    alpha=0.25, ax=axes[1]
)
lower = min(float(y_test.min()), float(best_prediction.min()))
upper = max(float(y_test.max()), float(best_prediction.max()))
axes[1].plot([lower, upper], [lower, upper], "k--")
axes[1].set(
    xlabel="Actual GPA change",
    ylabel="Predicted GPA change",
    title=f"Actual vs predicted: {best_model_name}",
)
fig.tight_layout()
plt.show()

,Model,Test MAE,Test RMSE,Test R2
0,Tuned histogram gradient boosting,0.1112,0.1414,0.4185
1,Histogram gradient boosting,0.1114,0.1416,0.4166
2,Random forest,0.1142,0.1448,0.3896
3,Linear regression,0.1242,0.1583,0.2712
4,Ridge regression,0.1242,0.1583,0.2712
5,Mean baseline,0.1444,0.1854,-0.0003


## 9. Residual and error analysis

A residual is `actual - predicted`. Residuals near zero are good. A pattern or strong skew suggests that the model misses part of the relationship.

In [10]:
residuals = np.asarray(y_test) - np.asarray(best_prediction)
absolute_errors = np.abs(residuals)

residual_summary = pd.Series({
    "Mean residual": residuals.mean(),
    "Residual standard deviation": residuals.std(ddof=1),
    "Median absolute error": np.median(absolute_errors),
    "Predictions within 0.10 GPA points": (absolute_errors <= 0.10).mean(),
    "Predictions within 0.20 GPA points": (absolute_errors <= 0.20).mean(),
})
display(residual_summary.to_frame("Value").style.format("{:.4f}"))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(x=best_prediction, y=residuals, alpha=0.25, ax=axes[0])
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set(
    xlabel="Predicted GPA change",
    ylabel="Residual",
    title="Residuals versus predictions",
)
sns.histplot(residuals, bins=40, kde=True, ax=axes[1])
axes[1].axvline(0, color="black", linestyle="--")
axes[1].set_title("Residual distribution")
fig.tight_layout()
plt.show()

,Value
Mean residual,0.0019
Residual standard deviation,0.1414
Median absolute error,0.0922
Predictions within 0.10 GPA points,0.5357
Predictions within 0.20 GPA points,0.8434


## 10. Feature importance

Permutation importance measures how much test performance worsens when one feature is shuffled. Larger values mean the feature was more useful to the selected model.

In [11]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_results = (
    pd.DataFrame({
        "Feature": X_test.columns,
        "Importance": importance.importances_mean,
        "Standard deviation": importance.importances_std,
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)
display(importance_results.style.format({
    "Importance": "{:.4f}",
    "Standard deviation": "{:.4f}",
}))

plt.figure(figsize=(9, 6))
sns.barplot(
    data=importance_results.head(10),
    y="Feature", x="Importance", color="#F58518"
)
plt.title("Top permutation importances")
plt.tight_layout()
plt.show()

,Feature,Importance,Standard deviation
0,Traditional_Study_Hours,0.0328,0.0010
1,Primary_Use_Case,0.0267,0.0008
2,Weekly_GenAI_Hours,0.0174,0.0004
3,Year_of_Study,0.0136,0.0003
4,Prompt_Engineering_Skill,0.0122,0.0002
5,Pre_Semester_GPA,0.0108,0.0003
6,Institutional_Policy,0.0014,0.0001
7,Paid_Subscription,0.0008,0.0001
8,Tool_Diversity,0.0000,0.0000
9,Perceived_AI_Dependency,0.0000,0.0000


## 11. Results analysis

The following interpretation is generated from the actual model outputs above. It states what won, whether AI variables added predictive value, how large the remaining errors were, and what the result can legitimately mean.

In [12]:
best_test_row = test_results.iloc[0]
baseline_test_row = test_results.loc[
    test_results["Model"] == "Mean baseline"
].iloc[0]
best_cv_row = cv_results.iloc[0]

context_r2 = float(feature_set_results.loc["Context only", "Test R2"])
ai_only_r2 = float(feature_set_results.loc["AI variables only", "Test R2"])
full_r2 = float(feature_set_results.loc["Context + AI variables", "Test R2"])
incremental_r2 = full_r2 - context_r2

top_features = importance_results.head(5)["Feature"].tolist()
within_010 = float((absolute_errors <= 0.10).mean())
within_020 = float((absolute_errors <= 0.20).mean())
cv_test_gap = float(best_test_row["Test R2"] - best_cv_row["CV R2"])

analysis_text = f'''
### Model comparison

- **{best_model_name}** produced the lowest test RMSE ({best_test_row["Test RMSE"]:.4f}) and a test R² of {best_test_row["Test R2"]:.4f}.
- The mean baseline had RMSE {baseline_test_row["Test RMSE"]:.4f} and R² {baseline_test_row["Test R2"]:.4f}. The selected model therefore provides a meaningful improvement over predicting the same average change for everyone.
- The strongest untuned cross-validation result was **{best_cv_row["Model"]}**, with mean R² {best_cv_row["CV R2"]:.4f}. The difference between the selected model's test R² and that cross-validation value was {cv_test_gap:+.4f}, which indicates how closely test performance matched the training-fold estimate.

### Do AI variables help?

- The context-only model achieved test R² {context_r2:.4f}.
- AI variables alone achieved test R² {ai_only_r2:.4f}.
- Combining context and AI variables achieved test R² {full_r2:.4f}, an increase of {incremental_r2:+.4f} over context alone.
- This means the AI-related columns contain **additional predictive information** in this dataset. It does **not** mean AI usage caused the GPA changes.

### Error behaviour

- {within_010:.1%} of test predictions were within 0.10 GPA points of the actual change.
- {within_020:.1%} were within 0.20 GPA points.
- The mean residual was {residuals.mean():+.4f}. A value close to zero means the model was not consistently too high or too low overall.

### Most useful predictors

The five most useful features by permutation importance were: **{", ".join(top_features)}**. Importance describes predictive contribution, not causal influence.

### Overall interpretation

GPA change is moderately predictable, but a substantial amount remains unexplained. The model is suitable for comparing algorithms and studying predictive patterns. It is not suitable for claiming that increasing AI usage will increase or decrease a student's GPA.
'''
display(Markdown(analysis_text))

analysis_summary = {
    "best_model": best_model_name,
    "best_test_mae": float(best_test_row["Test MAE"]),
    "best_test_rmse": float(best_test_row["Test RMSE"]),
    "best_test_r2": float(best_test_row["Test R2"]),
    "baseline_test_rmse": float(baseline_test_row["Test RMSE"]),
    "baseline_test_r2": float(baseline_test_row["Test R2"]),
    "best_cv_model": str(best_cv_row["Model"]),
    "best_cv_r2": float(best_cv_row["CV R2"]),
    "context_test_r2": context_r2,
    "ai_only_test_r2": ai_only_r2,
    "full_test_r2": full_r2,
    "incremental_ai_r2": incremental_r2,
    "within_0_10": within_010,
    "within_0_20": within_020,
    "mean_residual": float(residuals.mean()),
    "top_features": top_features,
}
print("ANALYSIS_SUMMARY_JSON=" + json.dumps(analysis_summary))


### Model comparison

- **Tuned histogram gradient boosting** produced the lowest test RMSE (0.1414) and a test R² of 0.4185.
- The mean baseline had RMSE 0.1854 and R² -0.0003. The selected model therefore provides a meaningful improvement over predicting the same average change for everyone.
- The strongest untuned cross-validation result was **Histogram gradient boosting**, with mean R² 0.4080. The difference between the selected model's test R² and that cross-validation value was +0.0105, which indicates how closely test performance matched the training-fold estimate.

### Do AI variables help?

- The context-only model achieved test R² 0.2382.
- AI variables alone achieved test R² 0.1703.
- Combining context and AI variables achieved test R² 0.4170, an increase of +0.1788 over context alone.
- This means the AI-related columns contain **additional predictive information** in this dataset. It does **not** mean AI usage caused the GPA changes.

### Error behaviour

- 53.6% of test predictions were within 0.10 GPA points of the actual change.
- 84.3% were within 0.20 GPA points.
- The mean residual was +0.0019. A value close to zero means the model was not consistently too high or too low overall.

### Most useful predictors

The five most useful features by permutation importance were: **Traditional_Study_Hours, Primary_Use_Case, Weekly_GenAI_Hours, Year_of_Study, Prompt_Engineering_Skill**. Importance describes predictive contribution, not causal influence.

### Overall interpretation

GPA change is moderately predictable, but a substantial amount remains unexplained. The model is suitable for comparing algorithms and studying predictive patterns. It is not suitable for claiming that increasing AI usage will increase or decrease a student's GPA.


ANALYSIS_SUMMARY_JSON={"best_model": "Tuned histogram gradient boosting", "best_test_mae": 0.11119387315239335, "best_test_rmse": 0.14136665572557847, "best_test_r2": 0.41846967761479426, "baseline_test_rmse": 0.18540515992356443, "baseline_test_r2": -0.000281023473369002, "best_cv_model": "Histogram gradient boosting", "best_cv_r2": 0.4080162648590667, "context_test_r2": 0.23817440562079406, "ai_only_test_r2": 0.1702941288306299, "full_test_r2": 0.4169779887758607, "incremental_ai_r2": 0.17880358315506661, "within_0_10": 0.5357, "within_0_20": 0.8434, "mean_residual": 0.001867985404500299, "top_features": ["Traditional_Study_Hours", "Primary_Use_Case", "Weekly_GenAI_Hours", "Year_of_Study", "Prompt_Engineering_Skill"]}


## 12. Recommendations and limitations

### Recommendations

- Use the tuned nonlinear model only as a predictive benchmark, not an intervention rule.
- Focus future data collection on verified student cohorts, actual AI-use logs, course difficulty, assessment type, and longitudinal outcomes.
- Include a genuine non-AI or low-AI comparison group if the future goal is to estimate an effect.
- Investigate why the most important predictors help and whether the patterns reproduce in independently collected data.

### Limitations

- The dataset's collection method and real-versus-synthetic status are undocumented.
- Nearly all rows contain an AI use case and at least one AI tool, so there is no credible non-user control group.
- The dataset is perfectly clean, which may indicate simulated or heavily prepared data.
- GPA change may be affected by course difficulty, instructor, assessment type, prior ability, and other unavailable variables.
- Prediction and feature importance measure association, not causation.

## 13. Conclusion

This notebook created a complete GPA-change regression workflow and compared multiple models under consistent five-fold validation. It also tested whether AI-related variables added predictive value beyond academic and study context.

The final conclusion must be based on the executed results in Section 11. The correct claim is that AI variables provide additional predictive information in this dataset. The notebook cannot establish that AI usage caused the observed GPA changes.